# Job Postings — Exploratory Data Analysis

Explores `data/processed/jobs.parquet`, built by `python -m src.data_loader`.

The corpus is a reproducible sample of the LinkedIn Job Postings 2023-2024
dataset. Skills are extracted with the same spaCy `PhraseMatcher` and taxonomy
used on resumes, so both sides of the recommender share one vocabulary.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import plotly.express as px

# Allow importing from src/ when the notebook runs from notebooks/.
sys.path.insert(0, str(Path.cwd().parent))

from src.config import JOBS_PARQUET

TEMPLATE = "plotly_dark"

jobs = pd.read_parquet(JOBS_PARQUET)
print(f"{len(jobs):,} postings, {len(jobs.columns)} columns")
jobs.head(3)

## Dataset overview

In [ ]:
skill_counts = jobs["skills"].map(len)

overview = pd.Series({
    "Postings": f"{len(jobs):,}",
    "Companies": f"{jobs['company'].nunique():,}",
    "Locations": f"{jobs['location'].nunique():,}",
    "Distinct titles": f"{jobs['title'].nunique():,}",
    "Median description length": f"{int(jobs['description'].str.len().median()):,} chars",
    "Mean skills per posting": f"{skill_counts.mean():.1f}",
    "Postings with no skills": f"{(skill_counts == 0).sum():,} ({(skill_counts == 0).mean():.0%})",
    "Experience requirement stated": f"{jobs['min_years_experience'].notna().mean():.0%}",
    "Salary present": f"{jobs['salary'].notna().mean():.0%}",
}, name="value")

overview.to_frame()

## Most common job titles

In [ ]:
top_titles = jobs["title"].value_counts().head(20).sort_values()

px.bar(
    x=top_titles.values,
    y=top_titles.index,
    orientation="h",
    template=TEMPLATE,
    labels={"x": "postings", "y": ""},
    title="Top 20 job titles",
    height=600,
)

## Where the jobs are

In [ ]:
top_locations = jobs["location"].value_counts().head(20).sort_values()

px.bar(
    x=top_locations.values,
    y=top_locations.index,
    orientation="h",
    template=TEMPLATE,
    labels={"x": "postings", "y": ""},
    title="Top 20 locations",
    height=600,
)

## Description length

This drives the 1,500-character truncation in `build_job_text`: the informative
part of a posting sits at the top, while the tail is usually benefits and equal
opportunity boilerplate.

In [ ]:
lengths = jobs["description"].str.len()

fig = px.histogram(
    x=lengths.clip(upper=15000),
    nbins=60,
    template=TEMPLATE,
    labels={"x": "description length (characters, clipped at 15k)"},
    title="Description length distribution",
)
fig.add_vline(x=1500, line_dash="dash", annotation_text="embedding cutoff")
fig.show()

lengths.describe().round(0).to_frame("chars")

## The 30 most common extracted skills

In [ ]:
skill_frequency = Counter(skill for row in jobs["skills"] for skill in row)
top_skills = pd.Series(dict(skill_frequency.most_common(30))).sort_values()

px.bar(
    x=top_skills.values,
    y=top_skills.index,
    orientation="h",
    template=TEMPLATE,
    labels={"x": "postings mentioning the skill", "y": ""},
    title="30 most common extracted skills",
    height=800,
)

Soft skills dominate because they appear in postings of every kind, while a
given technical skill only appears in its own niche. The long tail is what
actually separates one posting from another during matching.

## Experience requirements

In [ ]:
stated = jobs["min_years_experience"].dropna()

px.histogram(
    x=stated,
    nbins=int(stated.max()),
    template=TEMPLATE,
    labels={"x": "minimum years of experience required"},
    title=f"Stated experience requirement ({len(stated):,} of {len(jobs):,} postings)",
)

In [ ]:
levels = jobs["experience_level"].fillna("Not stated").value_counts()

px.bar(
    x=levels.index,
    y=levels.values,
    template=TEMPLATE,
    labels={"x": "", "y": "postings"},
    title="Seniority label supplied by the dataset",
)